# Chart Test Notebook (Cross-Asset BTC Benchmark)

This notebook tests the updated `streamlit_app.py` chart logic:

1. Normalized cross-asset performance chart (Altair)
2. Individual asset vs BTC line chart (Altair)
3. Asset minus BTC delta area chart (Altair)
4. Key ratio table structure test

It uses deterministic synthetic data, mirrors app transformations, and avoids renderer-specific `.show()` calls.


In [7]:
import numpy as np
import pandas as pd
import altair as alt

np.random.seed(42)
BTC_LABEL = "Bitcoin (BTC-USD)"
PEERS = ["Ethereum (ETH-USD)", "Solana (SOL-USD)", "Apple (AAPL)", "S&P 500 (^GSPC)"]
ALL_ASSETS = [BTC_LABEL] + PEERS


In [8]:
# Synthetic daily price data (deterministic) for reproducible chart tests
n_days = 120
dates = pd.date_range("2025-01-01", periods=n_days, freq="D")

base_returns = {
    BTC_LABEL: np.random.normal(0.0008, 0.02, n_days),
    "Ethereum (ETH)": np.random.normal(0.0010, 0.025, n_days),
    "Solana (SOL)": np.random.normal(0.0014, 0.035, n_days),
    "Chainlink (LINK)": np.random.normal(0.0009, 0.03, n_days),
}

prices = {}
start_prices = {
    BTC_LABEL: 60000,
    "Ethereum (ETH)": 3000,
    "Solana (SOL)": 140,
    "Chainlink (LINK)": 14,
}

for coin, rets in base_returns.items():
    prices[coin] = start_prices[coin] * np.cumprod(1 + rets)

pivot = pd.DataFrame(prices, index=dates)
normalized = pivot / pivot.iloc[0]
relative_to_btc = normalized[PEERS].divide(normalized[BTC_LABEL], axis=0) - 1.0

pivot.head()


,Bitcoin (BTC),Ethereum (ETH),Solana (SOL),Chainlink (LINK)
2025-01-01,60644.056984,3062.327396,136.312648,14.230726
2025-01-02,60524.874066,2995.768671,135.956085,14.897893
2025-01-03,61357.319309,3103.825620,138.549387,14.862692
2025-01-04,63275.385749,2998.151915,142.941601,15.055184
2025-01-05,63029.683156,3045.137235,137.136689,15.380441


In [9]:
# Chart 1 test: normalized cross-asset performance (Altair)
line_data = normalized.reset_index().melt(
    id_vars=["index"], var_name="Asset", value_name="Normalized price"
).rename(columns={"index": "Date"})

line_chart = (
    alt.Chart(line_data)
    .mark_line()
    .encode(
        alt.X("Date:T"),
        alt.Y("Normalized price:Q").scale(zero=False),
        alt.Color("Asset:N"),
        strokeWidth=alt.condition(alt.datum.Asset == BTC_LABEL, alt.value(4), alt.value(2)),
    )
    .properties(height=350, title="Normalized performance (base = 1)")
)

assert line_data["Asset"].nunique() == len(ALL_ASSETS), "Unexpected asset count"
assert line_data.groupby("Asset").size().eq(n_days).all(), "Trace length mismatch"
chart_dict = line_chart.to_dict()
assert "encoding" in chart_dict and "mark" in chart_dict, "Invalid Altair spec"

print("OK: normalized cross-asset chart")
line_chart


OK: normalized Altair chart


alt.Chart(...)

In [10]:
# Chart 2 & 3 tests: per-coin vs BTC line + delta area charts (Altair)
pair_charts = {}
delta_charts = {}

for coin in PEERS:
    pair_df = pd.DataFrame(
        {
            "Date": normalized.index,
            coin: normalized[coin].values,
            "BTC": normalized[BTC_LABEL].values,
        }
    ).melt(id_vars=["Date"], var_name="Series", value_name="Price")

    pair_chart = (
        alt.Chart(pair_df)
        .mark_line()
        .encode(
            alt.X("Date:T"),
            alt.Y("Price:Q").scale(zero=False),
            alt.Color(
                "Series:N",
                scale=alt.Scale(domain=[coin, "BTC"], range=["red", "gray"]),
            ),
            alt.Tooltip(["Date", "Series", "Price"]),
        )
        .properties(title=f"{coin} vs BTC", height=300)
    )

    delta_df = pd.DataFrame(
        {
            "Date": normalized.index,
            "Delta": normalized[coin] - normalized[BTC_LABEL],
        }
    )
    delta_chart = (
        alt.Chart(delta_df)
        .mark_area()
        .encode(
            alt.X("Date:T"),
            alt.Y("Delta:Q").scale(zero=False),
            alt.Tooltip(["Date", "Delta"]),
        )
        .properties(title=f"{coin} minus BTC", height=300)
    )

    # Assertions
    assert pair_df["Series"].nunique() == 2, f"{coin}: pair chart should have 2 series"
    assert len(delta_df) == n_days, f"{coin}: delta length mismatch"
    assert "encoding" in pair_chart.to_dict(), f"{coin}: pair spec invalid"
    assert "encoding" in delta_chart.to_dict(), f"{coin}: delta spec invalid"

    pair_charts[coin] = pair_chart
    delta_charts[coin] = delta_chart

print("OK: per-coin Altair pair and delta charts")

# Display one example pair
pair_charts[PEERS[0]]
delta_charts[PEERS[0]]


OK: per-coin Altair pair and delta charts


alt.Chart(...)

In [11]:
# Ratio table structure test (same columns as app)
ratio_rows = []
for i, symbol in enumerate(["BTC-USD", "ETH-USD", "AAPL", "^GSPC"]):
    ratio_rows.append(
        {
            "Symbol": symbol,
            "Asset": ALL_ASSETS[i],
            "Market cap": 10_000_000_000 * (i + 1),
            "Trailing P/E": 10.0 + i,
            "Forward P/E": 9.5 + i,
            "Price/Book": 2.0 + i,
            "Beta": 0.8 + 0.1 * i,
            "Dividend yield": 0.01 * i,
            "52w high": 100 + 10 * i,
            "52w low": 60 + 7 * i,
        }
    )

ratios_df = pd.DataFrame(ratio_rows)
expected_cols = {
    "Symbol",
    "Asset",
    "Market cap",
    "Trailing P/E",
    "Forward P/E",
    "Price/Book",
    "Beta",
    "Dividend yield",
    "52w high",
    "52w low",
}
assert set(ratios_df.columns) == expected_cols, "Ratios table columns mismatch"
assert len(ratios_df) >= 4, "Ratios table row count mismatch"

print("OK: ratios table structure")
ratios_df


OK: summary table


,coin,return_vs_btc_%,return_%,btc_return_%,volatility_annual_%,sharpe_annual
0,Ethereum (ETH),50.617857,32.900213,-11.763309,48.451380,2.040436
2,Chainlink (LINK),45.840406,28.684749,-11.763309,61.349016,1.565090
1,Solana (SOL),35.010258,19.128584,-11.763309,60.943704,1.184501


In [12]:
# Optional live yfinance smoke test (small request budget)
import yfinance as yf

symbols = ["BTC-USD", "ETH-USD", "AAPL", "^GSPC"]
live = yf.download(
    tickers=symbols,
    period="3mo",
    interval="1d",
    auto_adjust=True,
    progress=False,
)

print("Columns top-level:", list(live.columns.levels[0]) if hasattr(live.columns, "levels") else list(live.columns))
close = live["Close"] if "Close" in live.columns.levels[0] else live
print("Close shape:", close.shape)
print(close.tail(3))


Status: 429
Body: {"status":{"error_code":429,"error_message":"You've exceeded the Rate Limit. Please visit https://www.coingecko.com/en/api/pricing to subscribe to our API plans for higher rate limits."}}


SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3285131926.py, line 3)

In [13]:
!curl -H "X-CMC_PRO_API_KEY: b54bcf4d-1bca-4e8e-9a24-22ff2c3d462c" -H "Accept: application/json" -d "start=1&limit=5000&convert=USD" -G https://sandbox-api.coinmarketcap.com/v1/cryptocurrency/listings/latest

{"status":{"timestamp":"2026-04-23T11:00:24.465Z","error_code":0,"error_message":null,"elapsed":1,"credit_count":1,"notice":null},"data":[{"id":4599,"name":"k43zn5ucf6","symbol":"uyywt9rear8","slug":"2uus79rwfo2","cmc_rank":7362,"num_market_pairs":6917,"circulating_supply":7788,"total_supply":7393,"max_supply":1329,"infinite_supply":null,"last_updated":"2026-04-23T11:00:24.464Z","date_added":"2026-04-23T11:00:24.464Z","tags":["22obvatvj6w","u6fjv3uizo","uqbwk238t5j","hqz73mxigr4","2l8itv7yo6c","55ex2yyxj1w","ahu6slrefxb","61gfpscs2ha","p83hct373lp","wln7g50zf3"],"platform":null,"self_reported_circulating_supply":null,"self_reported_market_cap":null,"minted_market_cap":0.3934138044766893,"quote":{"USD":{"price":0.9990519865603302,"volume_24h":750,"volume_change_24h":0.8308915162259887,"percent_change_1h":0.570244233300842,"percent_change_24h":0.0030365314049363157,"percent_change_7d":0.5225308039086554,"market_cap":0.6700051047494238,"market_cap_dominance":4356,"fully_diluted_market_cap

{
  "status": {
    "timestamp": "2026-04-23T11:00:24.465Z",
    "error_code": 0,
    "error_message": null,
    "elapsed": 1,
    "credit_count": 1,
    "notice": null
  },
  "data": [
    {
      "id": 4599,
      "name": "k43zn5ucf6",
      "symbol": "uyywt9rear8",
      "slug": "2uus79rwfo2",
      "cmc_rank": 7362,
      "num_market_pairs": 6917,
      "circulating_supply": 7788,
      "total_supply": 7393,
      "max_supply": 1329,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.464Z",
      "date_added": "2026-04-23T11:00:24.464Z",
      "tags": [
        "22obvatvj6w",
        "u6fjv3uizo",
        "uqbwk238t5j",
        "hqz73mxigr4",
        "2l8itv7yo6c",
        "55ex2yyxj1w",
        "ahu6slrefxb",
        "61gfpscs2ha",
        "p83hct373lp",
        "wln7g50zf3"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.3934138044766893,
      "quote": {
        "USD": {
          "price": 0.9990519865603302,
          "volume_24h": 750,
          "volume_change_24h": 0.8308915162259887,
          "percent_change_1h": 0.570244233300842,
          "percent_change_24h": 0.0030365314049363157,
          "percent_change_7d": 0.5225308039086554,
          "market_cap": 0.6700051047494238,
          "market_cap_dominance": 4356,
          "fully_diluted_market_cap": 0.44370015970069443,
          "last_updated": "2026-04-23T11:00:24.464Z"
        }
      }
    },
    {
      "id": 682,
      "name": "uoq4u8vlcr",
      "symbol": "rko7fgiuhep",
      "slug": "2j2gj2a1mw5",
      "cmc_rank": 4887,
      "num_market_pairs": 6060,
      "circulating_supply": 5164,
      "total_supply": 1848,
      "max_supply": 4280,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.464Z",
      "date_added": "2026-04-23T11:00:24.464Z",
      "tags": [
        "v1ntwwhurra",
        "27737eww2qu",
        "h11s8tpri9u",
        "de22vbh78kb",
        "xir23l9rn7d",
        "zsh59j62xy9",
        "xuimx0q7h5l",
        "sk7ibd5dz69",
        "fx578mu8p2c",
        "2l2n6gqhvgo"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.4794223429333637,
      "quote": {
        "USD": {
          "price": 0.5531110639011618,
          "volume_24h": 745,
          "volume_change_24h": 0.0460419158323786,
          "percent_change_1h": 0.6397359049919085,
          "percent_change_24h": 0.6823341877352254,
          "percent_change_7d": 0.9207125785089272,
          "market_cap": 0.38673807965781193,
          "market_cap_dominance": 6266,
          "fully_diluted_market_cap": 0.12434469010800497,
          "last_updated": "2026-04-23T11:00:24.464Z"
        }
      }
    },
    {
      "id": 9470,
      "name": "ruxukezrdl",
      "symbol": "5k3vpvmyqm",
      "slug": "oexlf5malqj",
      "cmc_rank": 945,
      "num_market_pairs": 8574,
      "circulating_supply": 4108,
      "total_supply": 8982,
      "max_supply": 4806,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.464Z",
      "date_added": "2026-04-23T11:00:24.464Z",
      "tags": [
        "nq35hifsglb",
        "ahmtbfmf3yb",
        "9kj0c3q7atf",
        "re24b5m3ddf",
        "ahikdv3svrf",
        "09rc68frvp3k",
        "yphqyu1yqy",
        "3dyqgseab5a",
        "4y4dbqg9ccs",
        "b1ki13k8r8e"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.11457424655227522,
      "quote": {
        "USD": {
          "price": 0.18249398349730295,
          "volume_24h": 620,
          "volume_change_24h": 0.7380090493362774,
          "percent_change_1h": 0.546731140827692,
          "percent_change_24h": 0.1339299778075882,
          "percent_change_7d": 0.5026897383208737,
          "market_cap": 0.8634344935183111,
          "market_cap_dominance": 6330,
          "fully_diluted_market_cap": 0.5954495553900996,
          "last_updated": "2026-04-23T11:00:24.465Z"
        }
      }
    },
    {
      "id": 2036,
      "name": "tzx0fz7kvx",
      "symbol": "2n9u95afdlb",
      "slug": "1lm0rtglztf",
      "cmc_rank": 5911,
      "num_market_pairs": 284,
      "circulating_supply": 6536,
      "total_supply": 4478,
      "max_supply": 3862,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.465Z",
      "date_added": "2026-04-23T11:00:24.465Z",
      "tags": [
        "ss7i87egqya",
        "oq3a2m8hsfp",
        "hsd49evvg2s",
        "at5hyiddqml",
        "i01m7dxxdb",
        "udn4f9wapqg",
        "9arhr581cs",
        "ozhnn7mhyng",
        "i39po3pv2hn",
        "1jghwto916j"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.1394931869083884,
      "quote": {
        "USD": {
          "price": 0.11872675602592064,
          "volume_24h": 269,
          "volume_change_24h": 0.6849687955670816,
          "percent_change_1h": 0.1560784048900028,
          "percent_change_24h": 0.8686668084963749,
          "percent_change_7d": 0.855583922274967,
          "market_cap": 0.5451354453880657,
          "market_cap_dominance": 8965,
          "fully_diluted_market_cap": 0.9534935476636155,
          "last_updated": "2026-04-23T11:00:24.465Z"
        }
      }
    },
    {
      "id": 6129,
      "name": "6sow40xilir",
      "symbol": "65sdhzy64d3",
      "slug": "lbe2fcgw4gh",
      "cmc_rank": 7289,
      "num_market_pairs": 4353,
      "circulating_supply": 6155,
      "total_supply": 5501,
      "max_supply": 7219,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.465Z",
      "date_added": "2026-04-23T11:00:24.465Z",
      "tags": [
        "zu08eb8kkbg",
        "zuefcyd5gvp",
        "mmi6wsyf8h",
        "3o22320m5md",
        "rpbopu4zk8m",
        "lkgrfq75rr",
        "i3fjzm42ka",
        "0ptv6f7z588",
        "2d9vd4cmv3l",
        "sbe18ri6qsk"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.9701060214373891,
      "quote": {
        "USD": {
          "price": 0.6000054667771626,
          "volume_24h": 2815,
          "volume_change_24h": 0.5745617460981218,
          "percent_change_1h": 0.2732625747711115,
          "percent_change_24h": 0.7297329214295445,
          "percent_change_7d": 0.30267616467448866,
          "market_cap": 0.42097540035065095,
          "market_cap_dominance": 4035,
          "fully_diluted_market_cap": 0.13682462999471823,
          "last_updated": "2026-04-23T11:00:24.465Z"
        }
      }
    },
    {
      "id": 8997,
      "name": "ahjkvinu109",
      "symbol": "0caxzcouqaz",
      "slug": "blg97z3jvov",
      "cmc_rank": 9100,
      "num_market_pairs": 6667,
      "circulating_supply": 1776,
      "total_supply": 9850,
      "max_supply": 7857,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.465Z",
      "date_added": "2026-04-23T11:00:24.465Z",
      "tags": [
        "jn4iko9799h",
        "ckaiy7qcc",
        "thi00zf6x9",
        "wp31uq92twf",
        "gqq12tnr7j",
        "ky0d60h49a",
        "ic3vrvxrj4s",
        "dnx9jwjhx7",
        "8zmy1iu96jh",
        "8fr7a9ux1f3"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.0034080758682257795,
      "quote": {
        "USD": {
          "price": 0.4983163193367386,
          "volume_24h": 1897,
          "volume_change_24h": 0.5180233472561471,
          "percent_change_1h": 0.24189550149960004,
          "percent_change_24h": 0.015841691521240486,
          "percent_change_7d": 0.8144922151122762,
          "market_cap": 0.04327128809931491,
          "market_cap_dominance": 8147,
          "fully_diluted_market_cap": 0.6552333218259792,
          "last_updated": "2026-04-23T11:00:24.465Z"
        }
      }
    },
    {
      "id": 6549,
      "name": "b1yx1gup91f",
      "symbol": "c20wdgl1c58",
      "slug": "mau8mzgouw",
      "cmc_rank": 6102,
      "num_market_pairs": 7921,
      "circulating_supply": 2079,
      "total_supply": 49,
      "max_supply": 3296,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.465Z",
      "date_added": "2026-04-23T11:00:24.465Z",
      "tags": [
        "hauenz30uzr",
        "a4irwu8lqrk",
        "o98xz5pgf7",
        "2rmy1b5m3l",
        "pbquc0g95js",
        "w6q1zpnvsjn",
        "y8n94d7sv0n",
        "vct4dar26z",
        "r7qlanv74ec",
        "cu4r5xnxl1i"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.18045421921883986,
      "quote": {
        "USD": {
          "price": 0.8481613008081015,
          "volume_24h": 8247,
          "volume_change_24h": 0.33086995719549317,
          "percent_change_1h": 0.96340303923685,
          "percent_change_24h": 0.5311166224517132,
          "percent_change_7d": 0.33205785059419957,
          "market_cap": 0.5347220125061074,
          "market_cap_dominance": 2896,
          "fully_diluted_market_cap": 0.1735216298437563,
          "last_updated": "2026-04-23T11:00:24.465Z"
        }
      }
    },
    {
      "id": 1742,
      "name": "dn45j7ocs0r",
      "symbol": "qs0x4tuukr",
      "slug": "pw486ll3nw8",
      "cmc_rank": 2846,
      "num_market_pairs": 4915,
      "circulating_supply": 3961,
      "total_supply": 8315,
      "max_supply": 6773,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.465Z",
      "date_added": "2026-04-23T11:00:24.465Z",
      "tags": [
        "spp44kprcdd",
        "g5qlrcnoibk",
        "47ziadoks3b",
        "dhd1wp489pj",
        "azeksbuf1lk",
        "tgstcon3ixj",
        "ksen6dey2p",
        "4f0epcod666",
        "a54v2y72qcg",
        "rstz9dlpnas"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.4269866798361446,
      "quote": {
        "USD": {
          "price": 0.1556908558849528,
          "volume_24h": 6567,
          "volume_change_24h": 0.04169804109776076,
          "percent_change_1h": 0.3224663661082334,
          "percent_change_24h": 0.7388234012800377,
          "percent_change_7d": 0.5992193775176078,
          "market_cap": 0.8175650784070625,
          "market_cap_dominance": 6206,
          "fully_diluted_market_cap": 0.9926620356462676,
          "last_updated": "2026-04-23T11:00:24.465Z"
        }
      }
    },
    {
      "id": 7765,
      "name": "ml1s56lciyh",
      "symbol": "3x46yzswyp7",
      "slug": "gao9wy6t6bt",
      "cmc_rank": 7172,
      "num_market_pairs": 1428,
      "circulating_supply": 4090,
      "total_supply": 6827,
      "max_supply": 233,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.465Z",
      "date_added": "2026-04-23T11:00:24.465Z",
      "tags": [
        "ue5e8mu7uwd",
        "isug17lo4y",
        "e4c83l7qwr",
        "b0z5lxgqwzh",
        "f7dl66tmx9p",
        "anawd7thaev",
        "kl4kguvmah",
        "2xrbju9lola",
        "ru9t3p9m24g",
        "ozsjq83mzfd"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.1718557326693353,
      "quote": {
        "USD": {
          "price": 0.5523104775464664,
          "volume_24h": 9846,
          "volume_change_24h": 0.567982897191796,
          "percent_change_1h": 0.5084954422651733,
          "percent_change_24h": 0.2993175237998651,
          "percent_change_7d": 0.9571517983191757,
          "market_cap": 0.7068828406371368,
          "market_cap_dominance": 3085,
          "fully_diluted_market_cap": 0.6941172909566584,
          "last_updated": "2026-04-23T11:00:24.465Z"
        }
      }
    },
    {
      "id": 7997,
      "name": "qda7to82v6d",
      "symbol": "t8nquhhvbd",
      "slug": "kfs439wdma",
      "cmc_rank": 9771,
      "num_market_pairs": 9851,
      "circulating_supply": 8945,
      "total_supply": 9046,
      "max_supply": 7468,
      "infinite_supply": null,
      "last_updated": "2026-04-23T11:00:24.465Z",
      "date_added": "2026-04-23T11:00:24.465Z",
      "tags": [
        "f8axig1ixug",
        "y0pptrhbx0e",
        "9x8cy494b7k",
        "1u7oj4z9g4b",
        "svfoh78g8bb",
        "fxlll32s3b8",
        "geg88h8uiv",
        "sye1nb9hclj",
        "ccga1yzmewd",
        "qqvwpfo43m"
      ],
      "platform": null,
      "self_reported_circulating_supply": null,
      "self_reported_market_cap": null,
      "minted_market_cap": 0.9823575106049374,
      "quote": {
        "USD": {
          "price": 0.4937324799494611,
          "volume_24h": 4005,
          "volume_change_24h": 0.40380090462751017,
          "percent_change_1h": 0.7408007068959583,
          "percent_change_24h": 0.0769849760968615,
          "percent_change_7d": 0.92366698495871,
          "market_cap": 0.18328378404096246,
          "market_cap_dominance": 2506,
          "fully_diluted_market_cap": 0.02586619943073032,
          "last_updated": "2026-04-23T11:00:24.465Z"
        }
      }
    }
  ]
}